**Review**

Inconsistencies:

1. Add **evaluation megtric from Kaggle**: https://www.kaggle.com/competitions/ts-forecasting/overview

2.  **Leakage**: in load_feature_set, the scaler is fit globally across all training groups:

pythonscaler = RobustScaler().fit(all_train)

global scaler's median/IQR is computed from all series pooled together, which leaks distribution information across groups AND it is bad for sequential requirement

3.  The target column y_target is used as the first input feature

pythonmodel_cols = [TARGET_COL] + feature_cols   # y_target is channel 0

At position i in the sequence, you include y_target at time steps i through i+SEQ_LEN-1, and predict y_target at step i+SEQ_LEN. This is fine during training since those past targets are known. But at test time, the competition's test rows are future observations — you don't have the historical y_target values for the test group unless you reconstruct them from the training tail. **This is exactly what group_tails.pkl is meant to solve, but Block B/C/D never update or save it for the new feature sets.**


# CNN Experiments — Blocks B, C, D

This notebook assumes Block A (baseline CNN) has already been run and the following files have been saved:
- `best_cnn.pt` -- best model weights
- `scaler.pkl` -- fitted scaler
- `group_tails.pkl` -- group tails for inference
- `X_train, y_train, X_val, y_val` -- training and validation arrays in memory

Each block changes **one variable** at a time; everything else is kept fixed.

## Table of Contents
- [0 - Imports and Shared Utilities](#0)
- [1 - Block B: Feature Sets](#1)
- [2 - Block C: Training Hyperparameters](#2)
    - [2.1 - C1: Optimizer](#2-1)
    - [2.2 - C2: Learning Rate](#2-2)
    - [2.3 - C3: Batch Size](#2-3)
    - [2.4 - C4: LR Scheduler](#2-4)
    - [2.5 - C5: Early Stopping Patience](#2-5)
- [3 - Block D: Architecture Depth and Regularization](#3)
    - [3.1 - D1: Number of Layers](#3-1)
    - [3.2 - D2: Number of Filters](#3-2)
    - [3.3 - D3: Dropout](#3-3)
    - [3.4 - D4: Batch Normalization](#3-4)
- [4 - Summary Table](#4)

<a name="0"></a>
## 0 - Imports and Shared Utilities

In [1]:
# В Jupyter / VS Code / Cursor: встраивать графики в вывод ячейки (иначе иногда «пустой» рисунок)
%matplotlib inline

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import copy, time, warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: " + str(DEVICE))


Device: cpu


In [2]:
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Device: " + str(DEVICE))


Device: mps


In [3]:
# Shared constants (must match Block A)
INPUT_SIZE  = 30   # 1 target + 29 features
SEQ_LEN     = 20
EPOCHS      = 40   # max; early stopping will cut it short
PATIENCE    = 6

# Block A best config — carried forward as fixed baseline for B, C, D
BEST_A_NUM_FILTERS  = 64
BEST_A_KERNEL_SIZE  = 3
BEST_A_NUM_LAYERS   = 3
BEST_A_DROPOUT      = 0.2
BEST_A_BATCH_SIZE   = 256
BEST_A_LR           = 1e-3
BEST_A_OPTIMIZER    = "Adam"
BEST_A_SCHEDULER    = "ReduceOnPlateau"
BEST_A_FEATURE_SET  = "set_global_uncorr"  # placeholder; Block B picks the winner


In [4]:
# Experiment log — every run appends one row, never overwrite
results_log = []   # list of dicts → DataFrame at the end

def log_run(run_id, block, model_name, feature_set, optimizer_name,
            lr, batch_size, epochs_run, val_mae, val_rmse, extra=""):
    results_log.append({
        "run_id":      run_id,
        "block":       block,
        "model":       model_name,
        "feature_set": feature_set,
        "optimizer":   optimizer_name,
        "lr":          lr,
        "batch_size":  batch_size,
        "epochs_run":  epochs_run,
        "val_mae":     round(val_mae, 5),
        "val_rmse":    round(val_rmse, 5),
        "extra":       extra,
    })
    print(f"[{run_id}] MAE={val_mae:.5f}  RMSE={val_rmse:.5f}  epochs={epochs_run}")

def show_log():
    df = pd.DataFrame(results_log)
    display(df.sort_values(["block", "val_mae"]).reset_index(drop=True))
    return df


In [5]:
# Model definition (same as Block A)
class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size, dropout):
        """
        Single residual block with two Conv1d layers and BatchNorm.

        Arguments:
        channels -- number of input and output channels, integer
        kernel_size -- convolutional kernel size, integer
        dropout -- dropout probability, scalar
        """
        super().__init__()
        pad = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class CNNForecaster(nn.Module):
    def __init__(self, input_size=INPUT_SIZE, num_filters=64,
                 kernel_size=3, num_layers=3, dropout=0.2):
        """
        1D CNN forecaster with residual blocks and global average pooling.

        Arguments:
        input_size -- number of input channels (features), integer
        num_filters -- number of convolutional filters per layer, integer
        kernel_size -- convolutional kernel size, integer
        num_layers -- number of residual blocks, integer
        dropout -- dropout probability, scalar
        """
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(input_size, num_filters, kernel_size=1),
            nn.BatchNorm1d(num_filters),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(*[
            ResidualBlock(num_filters, kernel_size, dropout)
            for _ in range(num_layers)
        ])
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(num_filters, num_filters // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(num_filters // 2, 1),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.blocks(x)
        x = self.gap(x).squeeze(-1)
        return self.head(x).squeeze(-1)


class TSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


In [6]:
# Generic train + eval function
def make_optimizer(name, params, lr, weight_decay=1e-5):
    """
    Instantiate an optimizer by name.

    Arguments:
    name -- optimizer name, string ("Adam", "SGD", "RMSProp")
    params -- model parameters to optimize
    lr -- learning rate, scalar
    weight_decay -- L2 regularization coefficient, scalar

    Returns:
    optimizer -- torch.optim optimizer instance
    """
    if name == "Adam":     return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "SGD":      return torch.optim.SGD(params,  lr=lr, weight_decay=weight_decay, momentum=0.9)
    if name == "RMSProp":  return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")


def make_scheduler(name, optimizer):
    """
    Instantiate a learning rate scheduler by name.

    Arguments:
    name -- scheduler name, string ("ReduceOnPlateau", "StepLR", "CosineAnnealing", "None")
    optimizer -- optimizer instance to attach the scheduler to

    Returns:
    scheduler -- torch.optim.lr_scheduler instance, or None
    """
    if name == "ReduceOnPlateau":   return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", factor=0.5, patience=3)
    if name == "StepLR":            return torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    if name == "CosineAnnealing":   return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    if name == "None":              return None
    raise ValueError(f"Unknown scheduler: {name}")


def train_and_eval(model, train_loader, val_loader,
                   optimizer_name="Adam", lr=1e-3,
                   scheduler_name="ReduceOnPlateau",
                   patience=PATIENCE, epochs=EPOCHS,
                   save_path="best_exp.pt"):
    """
    Train the model and evaluate on validation set after each epoch.

    Arguments:
    model -- CNNForecaster instance to train
    train_loader -- DataLoader for the training set
    val_loader -- DataLoader for the validation set
    optimizer_name -- name of the optimizer to use, string ("Adam", "SGD", "RMSProp")
    lr -- learning rate, scalar
    scheduler_name -- name of the LR scheduler, string
    patience -- early stopping patience, integer
    epochs -- maximum number of training epochs, integer
    save_path -- file path to save the best model weights, string

    Returns:
    best_mae -- best validation MAE achieved, scalar
    best_rmse -- best validation RMSE achieved, scalar
    epochs_run -- number of epochs completed, integer
    history -- dictionary containing lists of train loss, val loss, and val MAE
    """
    if len(train_loader) == 0:
        raise RuntimeError("train_loader has 0 batches — no training rows. Check load_feature_set / N_SAMPLE_GROUPS.")
    if len(val_loader) == 0:
        raise RuntimeError("val_loader has 0 batches — cannot validate.")

    criterion = nn.HuberLoss(delta=1.0)
    optimizer = make_optimizer(optimizer_name, model.parameters(), lr)
    scheduler = make_scheduler(scheduler_name, optimizer)

    best_val_loss = float("inf")
    best_mae = best_rmse = float("inf")
    patience_ctr = 0
    history = {"train": [], "val": [], "mae": []}
    epoch_run = 0

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        train_loss_total, n = 0.0, 0
        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), Y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_total += loss.item() * len(Y_batch); n += len(Y_batch)
        train_loss = train_loss_total / n

        # Val
        model.eval()
        val_loss_total, n = 0.0, 0
        predictions_list, targets_list = [], []
        with torch.no_grad():
            for X_batch, Y_batch in val_loader:
                X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
                p = model(X_batch)
                val_loss_total += criterion(p, Y_batch).item() * len(Y_batch); n += len(Y_batch)
                predictions_list.append(p.cpu()); targets_list.append(Y_batch.cpu())
        val_loss = val_loss_total / n
        predictions = torch.cat(predictions_list); targets = torch.cat(targets_list)
        val_mae  = torch.mean(torch.abs(predictions - targets)).item()
        val_rmse = torch.sqrt(torch.mean((predictions - targets) ** 2)).item()

        # Scheduler step
        if scheduler is not None:
            if scheduler_name == "ReduceOnPlateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history["train"].append(train_loss)
        history["val"].append(val_loss)
        history["mae"].append(val_mae)
        epoch_run = epoch

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_mae  = val_mae
            best_rmse = val_rmse
            patience_ctr = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_ctr += 1

        if epoch % 5 == 0 or patience_ctr >= patience:
            print(f"  ep {epoch:3d} | train={train_loss:.5f} val={val_loss:.5f} MAE={val_mae:.5f}")

        if patience_ctr >= patience:
            print(f"  → Early stop at epoch {epoch}")
            break

    return best_mae, best_rmse, epoch_run, history


def plot_history(history, title):
    n = len(history.get("train", []))
    if n == 0:
        print(
            f"[plot_history] Нет точек для «{title}»: история обучения пуста. "
            "Обычно это значит, что до train_and_eval не дошло ни одной эпохи "
            "(ошибка выше, EPOCHS=0, или ячейка с графиком выполнена без обучения / не тем hist)."
        )
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(history["train"], label="train"); axes[0].plot(history["val"], label="val")
    axes[0].set_title(f"{title} — Huber loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history["mae"], color="orange", label="val MAE")
    axes[1].set_title(f"{title} — Val MAE"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


<a name="1"></a>
## 1 - Config: Horizons & Data Paths

Each horizon (1, 3, 10, 25) has its own dedicated feature set. Blocks C and D
are run independently for every horizon; the best config per horizon is used for
final training and test inference.

| Horizon | Feature set file |
|---------|-----------------|
| 1  | `data/set_h1_uncorr.parquet`  |
| 3  | `data/set_h3_uncorr.parquet`  |
| 10 | `data/set_h10_uncorr.parquet` |
| 25 | `data/set_h25_uncorr.parquet` |


In [ ]:
# ── Horizon → preprocessed data folder mapping ───────────────────────────────
# Each folder contains: train.parquet, valid.parquet, test.parquet
# train/valid have: id, code, sub_code, sub_category, horizon, ts_index,
#                   weight, split, y_target, feature_*
# test has the same columns minus y_target, weight, split
HORIZONS = {
    1:  "/Users/petyaeva.en/Downloads/final_feature_sets_split/set_h1_uncorr",
    3:  "/Users/petyaeva.en/Downloads/final_feature_sets_split/set_h3_uncorr",
    10: "/Users/petyaeva.en/Downloads/final_feature_sets_split/set_h10_uncorr",
    25: "/Users/petyaeva.en/Downloads/final_feature_sets_split/set_h25_uncorr",
}

# Original test.parquet — used ONLY to recover the official submission row order
TEST_PATH_ORIG = "data/test.parquet"

ID_COL     = "id"
GROUP_KEYS = ["code", "sub_code", "sub_category", "horizon"]
TARGET_COL = "y_target"
TIME_COL   = "ts_index"
N_SAMPLE_GROUPS = 36_000


In [ ]:
def load_feature_set_for_horizon(hz, folder, n_sample=N_SAMPLE_GROUPS, seq_len=SEQ_LEN):
    """
    Load preprocessed train.parquet + valid.parquet from `folder`.
    Fits RobustScaler on train. Returns sliding-window sequences + scaler.

    Parameters
    ----------
    hz       : int   horizon value (used only for logging)
    folder   : str   path to folder containing train.parquet / valid.parquet
    n_sample : int   max number of groups to use (for experiment speed);
                     pass a large number for final training
    seq_len  : int   sliding-window length

    Returns
    -------
    X_tr, y_tr, X_va, y_va : np.float32 arrays
    input_size              : int  (1 + n_features)
    scaler                  : fitted RobustScaler  (fit on train groups only)
    feature_cols            : list[str]  feature column names
    """
    from sklearn.preprocessing import RobustScaler
    import gc

    _skip = {"id", "code", "sub_code", "sub_category", "horizon",
             "ts_index", "weight", "split", "y_target"}

    df_tr = pd.read_parquet(f"{folder}/train.parquet")
    df_va = pd.read_parquet(f"{folder}/valid.parquet")

    feature_cols = [c for c in df_tr.columns if c not in _skip]
    model_cols   = [TARGET_COL] + feature_cols
    input_size   = 1 + len(feature_cols)
    print(f"  h{hz} | train={len(df_tr):,} val={len(df_va):,} | {input_size} channels ({len(feature_cols)} features)")

    # Sample a subset of groups for experiment speed
    unique_keys  = df_tr[GROUP_KEYS].drop_duplicates()
    sampled_keys = unique_keys.sample(n=min(n_sample, len(unique_keys)), random_state=SEED)
    df_tr = df_tr.merge(sampled_keys, on=GROUP_KEYS, how="inner")
    df_va = df_va.merge(sampled_keys, on=GROUP_KEYS, how="inner")

    def _build_groups(df):
        groups = {}
        for name, gdf in df.groupby(GROUP_KEYS, sort=False):
            gdf = gdf.sort_values(TIME_COL)
            gdf[model_cols] = gdf[model_cols].ffill().bfill().fillna(0.0)
            vals = gdf[model_cols].values.astype(np.float32)
            if len(vals) >= seq_len + 1:
                groups[name] = vals
        return groups

    group_tr = _build_groups(df_tr)
    group_va = _build_groups(df_va)
    del df_tr, df_va; gc.collect()

    # Fit scaler on training data only
    all_train = np.concatenate(list(group_tr.values()))
    scaler    = RobustScaler().fit(all_train)
    del all_train; gc.collect()

    def _make_seqs(groups):
        Xs, ys = [], []
        for vals in groups.values():
            s = scaler.transform(vals).astype(np.float32)
            T, F = s.shape
            N    = T - seq_len
            if N <= 0:
                continue
            X = np.empty((N, F, seq_len), dtype=np.float32)
            y = np.empty(N,              dtype=np.float32)
            for i in range(N):
                X[i] = s[i:i+seq_len].T
                y[i] = s[i+seq_len, 0]
            Xs.append(X); ys.append(y)
        return np.concatenate(Xs), np.concatenate(ys)

    X_tr, y_tr = _make_seqs(group_tr)
    X_va, y_va = _make_seqs(group_va)
    print(f"  Train seqs: {len(X_tr):,} | Val seqs: {len(X_va):,}")
    return X_tr, y_tr, X_va, y_va, input_size, scaler, feature_cols


In [9]:
def make_loaders(X_tr, y_tr, X_va, y_va, batch_size):
    """Build train and validation DataLoaders."""
    tr = DataLoader(TSDataset(X_tr, y_tr), batch_size=batch_size,
                    shuffle=True, num_workers=0)
    va = DataLoader(TSDataset(X_va, y_va), batch_size=batch_size * 2,
                    num_workers=0)
    return tr, va


<a name="2"></a>
## 2 - Hyperparameter & Architecture Search (per horizon)

For each horizon the following experiments are run sequentially (greedy one-at-a-time):

**Block C — Training hyperparameters:**

| Step | Variable | Candidates |
|------|----------|-----------|
| C1 | Optimizer | Adam / SGD / RMSProp |
| C2 | Learning rate | 1e-4 / 1e-3 / 5e-3 |
| C3 | Batch size | 32 / 64 / 128 |
| C4 | LR Scheduler | ReduceOnPlateau / CosineAnnealing / None |
| C5 | Patience | 4 / 6 / 10 |

**Block D — Architecture:**

| Step | Variable | Candidates |
|------|----------|-----------|
| D1 | Num layers | 1 / 2 / 3 |
| D2 | Num filters | 64 / 128 / 256 |
| D3 | Dropout | 0.0 / 0.2 / 0.5 |


In [10]:
def run_experiments_for_horizon(hz, fs_path):
    """
    Run full C1-C5 + D1-D3 hyperparameter search for one horizon.
    Logs every candidate to results_log.
    Returns best_cfg (dict), scaler, feature_cols.
    """
    import gc
    hz_label = f"h{hz}"
    print(f"\n{'='*60}")
    print(f"  HORIZON {hz}  |  {fs_path}")
    print(f"{'='*60}")

    X_tr, y_tr, X_va, y_va, input_size, scaler, feature_cols = \
        load_feature_set_for_horizon(hz, fs_path)

    # Running best (updated greedily)
    best = {
        "optimizer":   BEST_A_OPTIMIZER,
        "lr":          BEST_A_LR,
        "batch_size":  BEST_A_BATCH_SIZE,
        "scheduler":   BEST_A_SCHEDULER,
        "patience":    PATIENCE,
        "num_layers":  BEST_A_NUM_LAYERS,
        "num_filters": BEST_A_NUM_FILTERS,
        "dropout":     BEST_A_DROPOUT,
    }

    def fresh_model(num_filters=None, num_layers=None, dropout=None):
        torch.manual_seed(SEED)
        return CNNForecaster(
            input_size  = input_size,
            num_filters = num_filters  if num_filters  is not None else best["num_filters"],
            kernel_size = BEST_A_KERNEL_SIZE,
            num_layers  = num_layers   if num_layers   is not None else best["num_layers"],
            dropout     = dropout      if dropout       is not None else best["dropout"],
        ).to(DEVICE)

    def run_trial(run_id, model, opt, lr, bs, sched, pat, extra):
        tr_l, va_l = make_loaders(X_tr, y_tr, X_va, y_va, bs)
        mae, rmse, ep, _ = train_and_eval(
            model, tr_l, va_l,
            optimizer_name=opt, lr=lr, scheduler_name=sched, patience=pat,
            save_path=f"_tmp_{hz}.pt",
        )
        log_run(run_id, hz_label, "CNN", f"set_h{hz}_uncorr",
                opt, lr, bs, ep, mae, rmse, extra=extra)
        return mae

    # ── C1: Optimizer ─────────────────────────────────────────────────────────
    print(f"\n--- C1: Optimizer ---")
    c1 = {}
    for opt in ["Adam", "SGD", "RMSProp"]:
        print(f"  {opt} ...", end=" ", flush=True)
        c1[opt] = run_trial(f"{hz_label}-C1-{opt}", fresh_model(), opt,
                            best["lr"], best["batch_size"], best["scheduler"],
                            best["patience"], "C1-optimizer")
        print(f"MAE={c1[opt]:.5f}")
    best["optimizer"] = min(c1, key=c1.get)
    print(f"  -> best: {best['optimizer']}")

    # ── C2: Learning rate ─────────────────────────────────────────────────────
    print(f"\n--- C2: Learning rate ---")
    c2 = {}
    for lr in [1e-4, 1e-3, 5e-3]:
        print(f"  lr={lr} ...", end=" ", flush=True)
        c2[lr] = run_trial(f"{hz_label}-C2-{lr}", fresh_model(), best["optimizer"],
                           lr, best["batch_size"], best["scheduler"],
                           best["patience"], "C2-lr")
        print(f"MAE={c2[lr]:.5f}")
    best["lr"] = min(c2, key=c2.get)
    print(f"  -> best: {best['lr']}")

    # ── C3: Batch size ────────────────────────────────────────────────────────
    print(f"\n--- C3: Batch size ---")
    c3 = {}
    for bs in [32, 64, 128]:
        print(f"  bs={bs} ...", end=" ", flush=True)
        c3[bs] = run_trial(f"{hz_label}-C3-{bs}", fresh_model(), best["optimizer"],
                           best["lr"], bs, best["scheduler"],
                           best["patience"], "C3-batch")
        print(f"MAE={c3[bs]:.5f}")
    best["batch_size"] = min(c3, key=c3.get)
    print(f"  -> best: {best['batch_size']}")

    # ── C4: LR Scheduler ──────────────────────────────────────────────────────
    print(f"\n--- C4: Scheduler ---")
    c4 = {}
    for sched in ["ReduceOnPlateau", "CosineAnnealing", "None"]:
        print(f"  {sched} ...", end=" ", flush=True)
        c4[sched] = run_trial(f"{hz_label}-C4-{sched}", fresh_model(), best["optimizer"],
                              best["lr"], best["batch_size"], sched,
                              best["patience"], "C4-scheduler")
        print(f"MAE={c4[sched]:.5f}")
    best["scheduler"] = min(c4, key=c4.get)
    print(f"  -> best: {best['scheduler']}")

    # ── C5: Patience ──────────────────────────────────────────────────────────
    print(f"\n--- C5: Patience ---")
    c5 = {}
    for pat in [4, 6, 10]:
        print(f"  patience={pat} ...", end=" ", flush=True)
        c5[pat] = run_trial(f"{hz_label}-C5-{pat}", fresh_model(), best["optimizer"],
                            best["lr"], best["batch_size"], best["scheduler"],
                            pat, "C5-patience")
        print(f"MAE={c5[pat]:.5f}")
    best["patience"] = min(c5, key=c5.get)
    print(f"  -> best: {best['patience']}")

    # ── D1: Num layers ────────────────────────────────────────────────────────
    print(f"\n--- D1: Num layers ---")
    d1 = {}
    for nl in [1, 2, 3]:
        print(f"  layers={nl} ...", end=" ", flush=True)
        d1[nl] = run_trial(f"{hz_label}-D1-{nl}", fresh_model(num_layers=nl),
                           best["optimizer"], best["lr"], best["batch_size"],
                           best["scheduler"], best["patience"], f"D1-layers={nl}")
        print(f"MAE={d1[nl]:.5f}")
    best["num_layers"] = min(d1, key=d1.get)
    print(f"  -> best: {best['num_layers']}")

    # ── D2: Num filters ───────────────────────────────────────────────────────
    print(f"\n--- D2: Num filters ---")
    d2 = {}
    for nf in [64, 128, 256]:
        print(f"  filters={nf} ...", end=" ", flush=True)
        d2[nf] = run_trial(f"{hz_label}-D2-{nf}",
                           fresh_model(num_filters=nf, num_layers=best["num_layers"]),
                           best["optimizer"], best["lr"], best["batch_size"],
                           best["scheduler"], best["patience"], f"D2-filters={nf}")
        print(f"MAE={d2[nf]:.5f}")
    best["num_filters"] = min(d2, key=d2.get)
    print(f"  -> best: {best['num_filters']}")

    # ── D3: Dropout ───────────────────────────────────────────────────────────
    print(f"\n--- D3: Dropout ---")
    d3 = {}
    for drop in [0.0, 0.2, 0.5]:
        print(f"  dropout={drop} ...", end=" ", flush=True)
        d3[drop] = run_trial(f"{hz_label}-D3-{drop}",
                             fresh_model(num_layers=best["num_layers"],
                                         num_filters=best["num_filters"],
                                         dropout=drop),
                             best["optimizer"], best["lr"], best["batch_size"],
                             best["scheduler"], best["patience"], f"D3-dropout={drop}")
        print(f"MAE={d3[drop]:.5f}")
    best["dropout"] = min(d3, key=d3.get)
    print(f"  -> best: {best['dropout']}")

    print(f"\n  Best config for horizon {hz}:")
    for k, v in best.items():
        print(f"    {k:15s}: {v}")
    gc.collect()
    return best, scaler, feature_cols


In [ ]:
# ── Run experiments for all 4 horizons ───────────────────────────────────────
best_configs  = {}   # {hz: cfg dict}
exp_scalers   = {}   # {hz: RobustScaler}  (from experiment data sample)
exp_feat_cols = {}   # {hz: list of feature column names}

for _hz, _path in HORIZONS.items():
    best_configs[_hz], exp_scalers[_hz], exp_feat_cols[_hz] = \
        run_experiments_for_horizon(_hz, _path)

print("\n\n=== All horizons done ===")
show_log()
pd.DataFrame(results_log).to_csv("experiment_log.csv", index=False)
print("Saved experiment_log.csv")



  HORIZON 10  |  data/set_h10_uncorr.parquet
  h10 | 1,337,236 rows | 13 channels (12 features)
  Train seqs: 104,353 | Val seqs: 5,096

--- C1: Optimizer ---
  Adam ...   ep   5 | train=10.91032 val=11.06110 MAE=11.39381
  ep  10 | train=10.24299 val=11.21976 MAE=11.54241
  ep  15 | train=9.34542 val=10.81135 MAE=11.13648
  → Early stop at epoch 15
[h10-C1-Adam] MAE=11.12463  RMSE=56.18779  epochs=15
MAE=11.12463
  SGD ...   ep   5 | train=17.29981 val=16.26897 MAE=16.60724
  ep  10 | train=16.45600 val=15.33072 MAE=15.68273
  ep  15 | train=15.51878 val=14.18849 MAE=14.56451
  ep  20 | train=14.87474 val=13.81373 MAE=14.19013
  ep  25 | train=14.31657 val=12.85908 MAE=13.23629
  ep  30 | train=13.99169 val=13.04916 MAE=13.41434
  ep  35 | train=13.74477 val=12.76749 MAE=13.12455
  ep  40 | train=13.78891 val=12.28258 MAE=12.64583
[h10-C1-SGD] MAE=12.64583  RMSE=63.62369  epochs=40
MAE=12.64583
  RMSProp ...   ep   5 | train=10.92887 val=11.24310 MAE=11.58070
  ep  10 | train=10.0780

<a name="3"></a>
## 3 - Best Config per Horizon


In [ ]:
print("== Best configuration per horizon ==")
print(f"{'Horizon':>8} | {'Optimizer':>10} | {'LR':>7} | {'BS':>4} | "
      f"{'Scheduler':>18} | {'Pat':>3} | {'Layers':>6} | {'Filters':>7} | {'Drop':>5}")
print("-" * 90)
for hz, cfg in best_configs.items():
    print(f"{hz:>8} | {cfg['optimizer']:>10} | {cfg['lr']:>7} | {cfg['batch_size']:>4} | "
          f"{cfg['scheduler']:>18} | {cfg['patience']:>3} | {cfg['num_layers']:>6} | "
          f"{cfg['num_filters']:>7} | {cfg['dropout']:>5}")


<a name="4"></a>
## 4 - Final Training (Full Dataset, Best Config per Horizon)


In [ ]:
# ── E: Final training — one model per horizon, full dataset ──────────────────
import gc, time

final_models    = {}   # {hz: trained CNNForecaster}
final_scalers   = {}   # {hz: RobustScaler fit on all train data}
final_feat_cols = {}   # {hz: feature column names}

_t0_total = time.perf_counter()

for _hz, _fs_path in HORIZONS.items():
    _cfg = best_configs[_hz]
    print(f"\n{'='*60}")
    print(f"  FINAL TRAINING  |  horizon={_hz}")
    print(f"{'='*60}")
    _t1 = time.perf_counter()

    # Load ALL groups (no sampling)
    _X_tr, _y_tr, _X_va, _y_va, _input_size, _scaler, _feat_cols = \
        load_feature_set_for_horizon(_hz, _fs_path, n_sample=999_999)

    _tr_dl = DataLoader(TSDataset(_X_tr, _y_tr), batch_size=_cfg["batch_size"],
                        shuffle=True, num_workers=0)
    _va_dl = DataLoader(TSDataset(_X_va, _y_va), batch_size=_cfg["batch_size"] * 2,
                        num_workers=0)
    del _X_tr, _y_tr, _X_va, _y_va; gc.collect()

    _model = CNNForecaster(
        input_size  = _input_size,
        num_filters = _cfg["num_filters"],
        kernel_size = BEST_A_KERNEL_SIZE,
        num_layers  = _cfg["num_layers"],
        dropout     = _cfg["dropout"],
    ).to(DEVICE)
    torch.manual_seed(SEED)

    _save_path = f"best_final_h{_hz}.pt"
    _mae, _rmse, _ep, _hist = train_and_eval(
        _model, _tr_dl, _va_dl,
        optimizer_name = _cfg["optimizer"],
        lr             = _cfg["lr"],
        scheduler_name = _cfg["scheduler"],
        patience       = _cfg["patience"],
        save_path      = _save_path,
    )

    # Reload best weights, store
    _model.load_state_dict(torch.load(_save_path, map_location=DEVICE, weights_only=True))
    _model.eval()
    final_models[_hz]    = _model
    final_scalers[_hz]   = _scaler
    final_feat_cols[_hz] = _feat_cols

    print(f"\nSaved {_save_path}  |  MAE={_mae:.5f}  RMSE={_rmse:.5f}  "
          f"epochs={_ep}  [{time.perf_counter()-_t1:.1f}s]")
    plot_history(_hist, f"Final model h{_hz}")
    gc.collect()

print(f"\nAll 4 models trained in {(time.perf_counter()-_t0_total)/60:.1f} min")


<a name="5"></a>
## 5 - Inference on Test → submission.csv


In [ ]:
# ── E2: Inference on test → submission.csv ───────────────────────────────────
# For each horizon: load {folder}/test.parquet, build sequences, batch forward pass.
# Merge predictions by id onto the ORIGINAL test.parquet row order (unchanged).
import gc, time

_t0_inf = time.perf_counter()
print("Running inference …")

# Load original test to fix submission row order
df_test_orig = pd.read_parquet(TEST_PATH_ORIG, columns=["id", "horizon"])
print(f"Original test rows: {len(df_test_orig):,}")

preds_frames = []

for _hz, _folder in HORIZONS.items():
    print(f"\nHorizon {_hz} …", end=" ", flush=True)
    _model     = final_models[_hz]
    _scaler    = final_scalers[_hz]
    _feat_cols = final_feat_cols[_hz]
    _n_feats   = len(_feat_cols)
    _input_sz  = 1 + _n_feats

    # Load preprocessed test for this horizon
    df_h = pd.read_parquet(f"{_folder}/test.parquet")

    _missing = [c for c in _feat_cols if c not in df_h.columns]
    if _missing:
        raise KeyError(f"h{_hz}: missing columns: {_missing}")

    ids_h, preds_h = [], []

    for _gkey, _gdf in df_h.groupby(GROUP_KEYS, sort=False):
        _gdf   = _gdf.sort_values(TIME_COL)
        _ids   = _gdf[ID_COL].values
        T_test = len(_ids)

        # Build input matrix: y_target placeholder = 0, then features
        _feats  = (_gdf[_feat_cols]
                      .ffill().bfill().fillna(0.0)
                      .values.astype(np.float32))
        _y_ph   = np.zeros((T_test, 1), dtype=np.float32)
        _data   = np.hstack([_y_ph, _feats])          # (T_test, input_sz)

        # Scale using the scaler fit on training data
        _data_sc = _scaler.transform(_data).astype(np.float32)

        # Pad rows if group is shorter than SEQ_LEN
        if T_test < SEQ_LEN:
            _pad     = np.zeros((SEQ_LEN - T_test, _input_sz), dtype=np.float32)
            _data_sc = np.vstack([_pad, _data_sc])

        T_pad = len(_data_sc)

        # Build all SEQ_LEN windows for this group as one batch
        # Each row i gets window ending at position (T_pad - T_test + i + SEQ_LEN)
        _windows = np.stack(
            [_data_sc[T_pad - T_test + i : T_pad - T_test + i + SEQ_LEN].T
             for i in range(T_test)],
            axis=0,
        ).astype(np.float32)                           # (T_test, input_sz, SEQ_LEN)

        _model.eval()
        with torch.no_grad():
            _xt      = torch.tensor(_windows, dtype=torch.float32).to(DEVICE)
            _preds_s = _model(_xt).cpu().numpy()       # (T_test,) — scaled predictions

        # Inverse-transform: RobustScaler is per-column → use dummy row with zeros
        _dummy       = np.zeros((T_test, _input_sz), dtype=np.float32)
        _dummy[:, 0] = _preds_s
        _preds_real  = _scaler.inverse_transform(_dummy)[:, 0]

        ids_h.extend(_ids.tolist())
        preds_h.extend(_preds_real.tolist())

    preds_frames.append(pd.DataFrame({"id": ids_h, "y_target": preds_h}))
    print(f"{len(ids_h):,} rows predicted")
    gc.collect()

# Combine and align to ORIGINAL test.parquet row order
all_preds  = pd.concat(preds_frames, ignore_index=True)

# Sanity: check coverage
n_missing = all_preds["y_target"].isna().sum()
if len(all_preds) != len(df_test_orig):
    print(f"WARNING: predicted {len(all_preds)} rows, expected {len(df_test_orig)}")

submission = df_test_orig[["id"]].merge(all_preds, on="id", how="left")

if submission["y_target"].isna().any():
    n_nan = submission["y_target"].isna().sum()
    print(f"WARNING: {n_nan} NaN predictions after merge — check horizon coverage")

submission.to_csv("submission.csv", index=False)
print(f"\nSaved submission.csv  |  {len(submission):,} rows  "
      f"|  {time.perf_counter()-_t0_inf:.1f}s")
print(f"Preview (first 10):\n{submission.head(10).to_string(index=False)}")
print(f"\nStats: mean={submission.y_target.mean():.4f}  "
      f"std={submission.y_target.std():.4f}  "
      f"min={submission.y_target.min():.4f}  "
      f"max={submission.y_target.max():.4f}")

# Verify row order matches original test.parquet exactly
assert list(submission["id"]) == list(df_test_orig["id"]), \
    "ERROR: submission row order does not match original test.parquet!"
print("\nRow order verified: matches original test.parquet exactly.")
